# 03 — Local derivatives and chain rule

Part of the micrograd repetition pack.


## Goal
Attach the correct local derivative to each scalar operation. The key pattern is `input.grad += local_derivative * out.grad`.


In [ ]:
import math

_results = []

def check(name, condition, detail=""):
    ok = bool(condition)
    _results.append(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f" — {detail}" if detail else ""))

def close(a, b, tol=1e-6):
    return abs(a - b) <= tol

def summary():
    print(f"\nScore: {sum(_results)}/{len(_results)} tests passed")


In [ ]:
from math import exp, log, sin, cos

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data, self.grad = float(data), 0.0
        self._prev, self._op = set(_children), _op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            # TODO: addition local derivatives
            pass
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            # TODO: product local derivatives
            pass
        out._backward = _backward
        return out

    def exp(self):
        out = Value(exp(self.data), (self,), 'exp')
        def _backward():
            # TODO
            pass
        out._backward = _backward
        return out

    def log(self):
        out = Value(log(self.data), (self,), 'log')
        def _backward():
            # TODO
            pass
        out._backward = _backward
        return out

    def __pow__(self, power):
        out = Value(self.data ** power, (self,), f'**{power}')
        def _backward():
            # TODO
            pass
        out._backward = _backward
        return out

    def sin(self):
        out = Value(sin(self.data), (self,), 'sin')
        def _backward():
            # TODO
            pass
        out._backward = _backward
        return out


### Round A — Test each local rule in isolation
Setting `out.grad` manually represents the upstream gradient arriving from later in the graph.


In [ ]:
_results.clear()
try:
    a,b = Value(2),Value(3); out=a+b; out.grad=4; out._backward()
    check("add sends upstream grad to a", a.grad == 4)
    check("add sends upstream grad to b", b.grad == 4)
    a,b = Value(2),Value(3); out=a*b; out.grad=4; out._backward()
    check("multiply grad for a", a.grad == 12)
    check("multiply grad for b", b.grad == 8)
except Exception as e: check("binary local rules run", False, repr(e))
summary()


In [ ]:
_results.clear()
for name, make, expected in [
    ("exp", lambda x: x.exp(), math.exp(2)),
    ("log", lambda x: x.log(), 0.5),
    ("power", lambda x: x**3, 12.0),
    ("sin", lambda x: x.sin(), math.cos(2)),
]:
    try:
        x=Value(2); out=make(x); out.grad=1; out._backward()
        check(name, close(x.grad, expected))
    except Exception as e: check(name, False, repr(e))
summary()


### Round B — Chain two local rules manually
For `z=(x*y)+x`, call the local backward functions in reverse dependency order. Predict `dz/dx` and `dz/dy` first.


In [ ]:
_results.clear()
try:
    x,y=Value(2),Value(3)
    q=x*y
    z=q+x
    z.grad=1
    z._backward()
    q._backward()
    check("dz/dx includes both paths", x.grad == 4)
    check("dz/dy", y.grad == 2)
except Exception as e: check("manual chain rule", False, repr(e))
summary()


### Concept checks

1. In multiplication, why does `a.grad` receive `b.data * out.grad`?
2. Why must gradient contributions use `+=` instead of `=`?
3. Draw the two paths from `x` to `z` in `z=x*y+x`.
4. Reset the kernel and implement every rule again from memory.
